# v2 training — ALL models (B1–B4, P1, ablations) on Colab T4

**Rules (learned the hard way):**
1. `bank()` runs IMMEDIATELY after every model — three good v1 runs were lost to dead sessions.
2. One heavy model per session is fine; cells 1–3 must be rerun at the start of EVERY new session (they rebuild `/content`).
3. Run order: B1 → B2 → B3 → B4 → P1 → ablations (only after P1 looks sane) → optional P1-large.
4. v1 wall-clock references (same T4): B4 ≈ 834 s, P1 ≈ 620 s per run. Budget quota accordingly across the 2 accounts.

Everything banks to `MyDrive/Final_Reporing_Sentiment_Analysis/thesis_v2/`.

In [ ]:
# cell 1 — environment (rerun at the start of EVERY session; note the torchao line)
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf repo
!git clone -b fix/v2-reproducibility-foundation https://github.com/ruwini01/Sinhala_English_Code_Mixed_Sentiment_Analysis.git repo
%cd /content/repo/ml
!pip install -q peft fasttext
!pip uninstall -y -q torchao

import os, shutil
SAVE = "/content/drive/MyDrive/Final_Reporing_Sentiment_Analysis/thesis_v2"
for sub in ("results", "checkpoints", "tokenized"):
    os.makedirs(f"{SAVE}/{sub}", exist_ok=True)

def bank(*paths, sub="results"):
    """Copy artifacts to Drive IMMEDIATELY — sessions die without warning."""
    for p in paths:
        if os.path.isdir(p):
            shutil.copytree(p, f"{SAVE}/{sub}/{os.path.basename(p)}", dirs_exist_ok=True)
        elif os.path.exists(p):
            shutil.copy2(p, f"{SAVE}/{sub}/")
        else:
            print("MISSING (not banked):", p)
    print("banked ->", f"{SAVE}/{sub}:", ", ".join(os.path.basename(p) for p in paths))

In [ ]:
# cell 2 — HF token from Colab Secrets (key icon in left sidebar, name: HF_TOKEN)
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# cell 3 — data: raw csv -> preprocess -> tokenize (splits are LOCKED in the repo)
import hashlib, os
RAW = "data/raw/singlish_mixed_sentiment_complete.csv"
RAW_SHA_V21 = "5ca2952ebf1087dbc0704beb4c53306bf3e30ab201ff72a60171565a932ba221"
os.makedirs("data/raw", exist_ok=True)

DRIVE_RAW = f"{SAVE}/singlish_mixed_sentiment_complete.csv"
if os.path.exists(DRIVE_RAW):
    shutil.copy2(DRIVE_RAW, RAW)
else:
    from google.colab import files
    files.upload()                      # pick the raw CSV from your PC
    os.replace("singlish_mixed_sentiment_complete.csv", RAW)
    shutil.copy2(RAW, DRIVE_RAW)        # bank the raw file itself

sha = hashlib.sha256(open(RAW, "rb").read()).hexdigest()
assert sha == RAW_SHA_V21, f"WRONG RAW FILE — sha256 {sha[:16]}... != v2.1 (see ml/DATA.md)"
print("raw sha256 verified: v2.1")

!python -m src.preprocess.quarantine
!python -m src.preprocess.clean_text
!python -m src.preprocess.language_id
# make_splits NOT run: the locked v2 split ids (7112/1524/1524) are committed in the repo
!python -m src.preprocess.tokenize_cache
bank("data/processed/tokenized/train.pt", "data/processed/tokenized/val.pt",
     "data/processed/tokenized/test.pt", sub="tokenized")

## Baselines B1–B4
Each cell = one model + immediate bank. B1 is CPU-only (~1 min) but runs here so every result JSON carries the same environment.

In [ ]:
# cell 4 — B1: TF-IDF + Logistic Regression (~1 min, CPU)
!python -m src.train.run_tfidf_lr
bank("results/tfidf_lr.json")

In [ ]:
# cell 5 — B2: BiLSTM + fastText cc.si.300 (embeddings ~1.5 GB download first time)
import os
os.makedirs("data/embeddings", exist_ok=True)
EMB_GZ = f"{SAVE}/cc.si.300.bin.gz"
if os.path.exists(EMB_GZ):
    shutil.copy2(EMB_GZ, "data/embeddings/cc.si.300.bin.gz")
else:
    !wget -q -O data/embeddings/cc.si.300.bin.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.bin.gz
    shutil.copy2("data/embeddings/cc.si.300.bin.gz", EMB_GZ)   # bank for next time
!gunzip -kf data/embeddings/cc.si.300.bin.gz

!python -m src.train.run_bilstm_fasttext --fasttext data/embeddings/cc.si.300.bin
bank("results/bilstm_fasttext.json")

In [ ]:
# cell 6 — B3: mBERT full fine-tune (T4, ~15 min)
!python -m src.train.run_mbert_full
bank("results/mbert_full.json")

In [ ]:
# cell 7 — B4: XLM-R full fine-tune (T4, ~15 min; efficiency comparison target for P1)
!python -m src.train.run_xlmr_full
bank("results/xlmr_full.json")

## P1 — proposed model (XLM-R frozen + LoRA q,v + LID + SCL)
Check val macro-F1 looks sane vs B4 before spending quota on ablations.

In [ ]:
# cell 8 — P1 stabilized run (T4, ~10-12 min)
!python -m src.train.run_lora_scl_lid --epochs 18 --patience 4
bank("results/lora_scl_lid.json")
bank("checkpoints/lora_scl_lid.pt", sub="checkpoints")   # explainability stage needs this

## Ablations (run ONLY after P1 looks sane)
Seven runs ≈ 70–90 min total — split across sessions/accounts if quota bites. Each banks before the next starts.

In [ ]:
# cell 9 — ablation batch (each result banked before the next run starts)
ablations = [
    ("abl_no_scl",    "--id abl_no_scl --lam 0"),
    ("abl_no_lid",    "--id abl_no_lid --no-lid"),
    ("abl_lora_only", "--id abl_lora_only --lam 0 --no-lid"),
    ("abl_no_lora",   "--id abl_no_lora --no-lora --lr 2e-5"),
    ("abl_rank_4",    "--id abl_rank_4 --rank 4"),
    ("abl_rank_16",   "--id abl_rank_16 --rank 16"),
    ("abl_q_only",    "--id abl_q_only --targets query"),
]
for exp_id, flags in ablations:
    print(f"\n{'='*60}\n{exp_id}\n{'='*60}")
    !python -m src.train.run_lora_scl_lid {flags}
    bank(f"results/{exp_id}.json")

In [ ]:
# cell 10 — OPTIONAL score push: xlm-roberta-large + LoRA (fits T4 only because of LoRA, ~25 min)
!python -m src.train.run_lora_scl_lid --id p1_large --model xlm-roberta-large --batch-size 16 --lr 1e-4
bank("results/p1_large.json")
bank("checkpoints/p1_large.pt", sub="checkpoints")

In [ ]:
# cell 11 — end of session: sweep every result JSON to Drive + download to PC
import glob
jsons = glob.glob("results/*.json")
bank(*jsons)
!zip -q -j /content/v2_results.zip results/*.json
from google.colab import files
files.download("/content/v2_results.zip")   # unzip into ml/results/ locally, then commit